# Telco Customer Churn — Feature Engineering Notebook

This notebook loads the cleaned dataset and performs feature engineering
to prepare the data for modeling. Steps include:
- Creating tenure groups
- Identifying categorical and numeric columns
- Building a preprocessing pipeline with OneHotEncoder and StandardScaler
- Exporting the transformed feature matrix for modeling


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/cleaned_telco_churn.csv")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   int64  
 4   Dependents        7032 non-null   int64  
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   int64  
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   int64  


In [3]:
# Creating Tenure Buckets
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12', '12-24', '24-48', '48-72']
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerID        7032 non-null   object  
 1   gender            7032 non-null   object  
 2   SeniorCitizen     7032 non-null   int64   
 3   Partner           7032 non-null   int64   
 4   Dependents        7032 non-null   int64   
 5   tenure            7032 non-null   int64   
 6   PhoneService      7032 non-null   int64   
 7   MultipleLines     7032 non-null   object  
 8   InternetService   7032 non-null   object  
 9   OnlineSecurity    7032 non-null   object  
 10  OnlineBackup      7032 non-null   object  
 11  DeviceProtection  7032 non-null   object  
 12  TechSupport       7032 non-null   object  
 13  StreamingTV       7032 non-null   object  
 14  StreamingMovies   7032 non-null   object  
 15  Contract          7032 non-null   object  
 16  PaperlessBilling  7032 n

In [4]:
# Identifying Categorical and Numeric Columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

# Removing Target and customerID
numeric_cols.remove('Churn')
categorical_cols.remove('customerID')

categorical_cols, numeric_cols

(['gender',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaymentMethod'],
 ['SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure',
  'PhoneService',
  'PaperlessBilling',
  'MonthlyCharges',
  'TotalCharges'])

In [5]:
# Preserving Index
df['original_index'] = df.index

In [6]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# Preprocessing Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [8]:
# Fitting to Training Data
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'Partner', 'Dependents',
                                  'tenure', 'PhoneService', 'PaperlessBilling',
                                  'MonthlyCharges', 'TotalCharges']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'MultipleLines', 'InternetService',
                                  'OnlineSecurity', 'OnlineBackup',
                                  'DeviceProtection', 'TechSupport',
                                  'StreamingTV', 'StreamingMovies', 'Contract',
                                  'PaymentMethod'])])

In [9]:
# Transforming sets
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [10]:
# Get feature names from OneHotEncoder
ohe = preprocessor.named_transformers_['cat']
ohe_features = ohe.get_feature_names_out(categorical_cols)

# Combine numeric + encoded categorical names
feature_names = numeric_cols + list(ohe_features)

# Build DataFrame
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

X_train_df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,gender_Female,gender_Male,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,-0.439319,1.028312,1.529143,1.321816,0.327542,-1.21303,0.981556,1.659900,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,-0.439319,-0.972468,-0.653961,-0.267410,-3.053048,-1.21303,-0.971546,-0.562252,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.439319,1.028312,-0.653961,1.444064,0.327542,-1.21303,0.837066,1.756104,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,-0.439319,-0.972468,-0.653961,-1.204646,0.327542,-1.21303,0.641092,-0.908326,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.439319,1.028312,-0.653961,0.669826,-3.053048,-1.21303,-0.808787,-0.101561,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [11]:
df.to_csv("../data/cleaned/cleaned_telco_churn.csv", index=False)
X_train_df.to_csv("../data/engineered/X_train_engineered.csv", index=False)
X_test_df.to_csv("../data/engineered/XX_test_engineered.csv", index=False)
y_train.to_csv("../data/engineered/Xy_train.csv", index=False)
y_test.to_csv("../data/engineered/Xy_test.csv", index=False)

# Summary

- Created tenure buckets
- Identified categorical and numeric columns
- Built a scikit-learn preprocessing pipeline
- Applied OneHotEncoder and StandardScaler
- Exported engineered train/test datasets for modeling